In [1]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

In [3]:
data = pl.read_parquet("data/ELENA_beam_steering_simple_optimise.parquet").with_columns(pl.all().exclude(['Run Number','SC12_coinc*event_clock']).round(3))
print(data)

shape: (186, 31)
┌────────┬─────────────┬────────────┬────────────┬───┬──────────┬───────────┬──────────┬───────────┐
│ Run    ┆ H_offset_re ┆ V_offset_r ┆ H_angle_re ┆ … ┆ QDNE08P  ┆ QDNE08N   ┆ QDNE14P  ┆ QDNE14N   │
│ Number ┆ quested     ┆ equested   ┆ quested    ┆   ┆ ---      ┆ ---       ┆ ---      ┆ ---       │
│ ---    ┆ ---         ┆ ---        ┆ ---        ┆   ┆ f64      ┆ f64       ┆ f64      ┆ f64       │
│ i32    ┆ f64         ┆ f64        ┆ f64        ┆   ┆          ┆           ┆          ┆           │
╞════════╪═════════════╪════════════╪════════════╪═══╪══════════╪═══════════╪══════════╪═══════════╡
│ 492714 ┆ -10.0       ┆ 0.0        ┆ 0.0        ┆ … ┆ 3165.578 ┆ -3165.83  ┆ 4343.586 ┆ -4343.912 │
│ 492715 ┆ -17.914     ┆ -6.369     ┆ 3.832      ┆ … ┆ 3165.58  ┆ -3165.831 ┆ 4343.588 ┆ -4343.915 │
│ 492716 ┆ -3.952      ┆ 0.31       ┆ -0.898     ┆ … ┆ 3165.58  ┆ -3165.831 ┆ 4343.589 ┆ -4343.915 │
│ 492717 ┆ -17.119     ┆ -4.111     ┆ 3.008      ┆ … ┆ 3165.576 ┆ -3165.82

In [4]:
corrector_names= [
    'DHZE05R',
    'DVTE05T',
    'DHZE08R',
    'DVTE08T',
    'DHZE14R',
    'DVTE14T',
    'QFNE09P',
    'QFNE15P',
    'QDNE08P',
    'QDNE14P',
    ]

parameter_names = [
    'H_offset_mm',
    'V_offset_mm',
    'H_angle_mrad',
    'V_angle_mrad',
]

In [5]:
# for col in parameter_names:
#     data:pl.DataFrame = data.with_columns(pl.col(offset).diff(1).alias(f'delta_{offset}'))
data:pl.DataFrame = data.with_columns(pl.col(parameter_names+corrector_names).diff(1).name.prefix('delta_'))
print(data.select(parameter_names + [f'delta_{offset}' for offset in parameter_names]))

shape: (186, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ H_offset_m ┆ V_offset_m ┆ H_angle_mr ┆ V_angle_m ┆ delta_H_o ┆ delta_V_o ┆ delta_H_a ┆ delta_V_a │
│ m          ┆ m          ┆ ad         ┆ rad       ┆ ffset_mm  ┆ ffset_mm  ┆ ngle_mrad ┆ ngle_mrad │
│ ---        ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ f64        ┆ f64        ┆ f64        ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ -10.0      ┆ 0.0        ┆ 0.0        ┆ 0.0       ┆ null      ┆ null      ┆ null      ┆ null      │
│ -17.914    ┆ -6.369     ┆ 3.832      ┆ 2.874     ┆ -7.914    ┆ -6.369    ┆ 3.832     ┆ 2.874     │
│ -3.952     ┆ 0.31       ┆ -0.898     ┆ 6.006     ┆ 13.962    ┆ 6.679     ┆ -4.73     ┆ 3.132     │
│ -17.119    ┆ -4.111     ┆ 3.008      ┆ 6.821     ┆ -13.167   ┆ -4.421    

In [6]:
print('##DELTAS')
deltas = data.filter(pl.col('delta_V_offset_mm').is_not_nan()).select([f'delta_{param}' for param in parameter_names]).to_numpy()
deltas = np.hstack([deltas,np.ones((deltas.shape[0],1))])
print(str(deltas[:5][:])[:-1])
print('...')
print(str(deltas[-5:][:])[0:].replace('[',' ',1))

print('##CCORRECTORS')
correctors = data.filter(pl.col('delta_V_offset_mm').is_not_nan()).select([f'delta_{name}' for name in corrector_names]).to_numpy()
print(str(correctors[:5][:])[:-1])
print('...')
print(str(correctors[-5:][:])[0:].replace('[',' ',1))


##DELTAS
[[ -7.914  -6.369   3.832   2.874   1.   ]
 [ 13.962   6.679  -4.73    3.132   1.   ]
 [-13.167  -4.421   3.906   0.815   1.   ]
 [ -2.385  -7.913  -0.064  -0.808   1.   ]
 [  7.385  12.597  -7.223  -3.679   1.   ]
...
 [-2.135 -0.481 -0.018  0.019  1.   ]
 [-0.182  0.014 -0.01  -0.061  1.   ]
 [-6.958 -9.532 -6.364  0.     1.   ]
 [ 9.908  0.    10.    -2.424  1.   ]
 [ 0.     7.787  0.     7.322  1.   ]]
##CCORRECTORS
[[ 1.000000e-03  0.000000e+00  1.034095e+03 -6.880340e+02 -2.472350e+02
  -7.305230e+02  3.000000e-03  2.000000e-03  2.000000e-03  2.000000e-03]
 [ 0.000000e+00  1.000000e-03 -1.576403e+03 -6.909610e+02  1.425420e+02
   7.325280e+02  0.000000e+00  0.000000e+00  0.000000e+00  1.000000e-03]
 [-1.000000e-03  0.000000e+00  1.418902e+03 -2.062530e+02 -5.424600e+01
  -4.448000e+00 -4.000000e-03 -5.000000e-03 -4.000000e-03 -7.000000e-03]
 [ 4.000000e-03  2.000000e-03  1.628300e+02 -3.406600e+01  1.018160e+02
  -1.380041e+03  1.000000e-03  2.000000e-03  2.000000e-03  3

In [7]:
coefficients,residuals,rank,s = np.linalg.lstsq(deltas,correctors,rcond=1e-2)
print(coefficients)
print(residuals)
print(rank)
print(s)

[[ 6.61346185e-06  7.89513332e-06 -6.57640853e+01  5.13890775e-02
  -3.81001083e+01 -8.01982851e-01 -3.72056899e-05 -2.19117705e-05
  -2.30975026e-05 -2.25540695e-06]
 [-3.15189107e-05 -2.38948766e-05  2.15263490e+00  5.05749004e+00
  -1.30494143e+00  1.12230765e+02  1.91500869e-04  1.44215102e-04
   2.13185096e-04  2.10569546e-04]
 [-3.89504137e-05 -4.18461306e-05  1.11751494e+02 -1.13591268e+00
  -1.32271408e+02 -2.16230251e-01 -2.10524575e-04 -2.03806951e-04
  -1.78562583e-04 -2.51588802e-04]
 [-4.33755207e-06 -9.73622203e-06 -1.23805105e+01 -2.30979220e+02
  -4.14941751e+00 -3.61656798e+00 -2.43591954e-04 -2.24055313e-04
  -3.34167300e-04 -3.23958239e-04]
 [-3.27262256e-05 -2.13883726e-05  5.82803056e-01  1.06083557e-01
  -2.42267375e-01 -3.11233175e-02 -5.15157187e-04 -4.53910571e-04
  -5.17462797e-04 -5.92830501e-04]]
[4.82561536e-04 8.01285839e-04 8.03735931e+06 3.05759207e+05
 9.05772466e+06 1.24331005e+06 4.80166853e-03 3.72093993e-03
 5.31896516e-03 6.16339421e-03]
5
[80.9661

In [8]:
initial_state = data.head(1).select(corrector_names).to_numpy()[0]
print(initial_state)

print(data.select(pl.col('Run Number').min().alias('Run Number min'),pl.col('Run Number').max().alias('Run Number max')))
run_number = 492751
parameters = data.filter(pl.col("Run Number")==492751).select([f'delta_{name}' for name in parameter_names]).to_numpy()[0]
print(parameters)

actual_state = data.filter(pl.col("Run Number")==492751).select([f'delta_{name}' for name in corrector_names]).to_numpy()[0]
print(actual_state)


# print(coefficients.shape)
# print(coefficients[1:][:].shape)
calculated_state = residuals + parameters @ coefficients[1:]
print(calculated_state)
print(actual_state-calculated_state)

[ 150.047 -999.984  345.959 1469.647 -316.811 1436.057 3042.649 1700.898
 3165.578 4343.586]
shape: (1, 2)
┌────────────────┬────────────────┐
│ Run Number min ┆ Run Number max │
│ ---            ┆ ---            │
│ i32            ┆ i32            │
╞════════════════╪════════════════╡
│ 492714         ┆ 492944         │
└────────────────┴────────────────┘
[6.293 0.    0.    1.881]
[ 0.00000e+00  0.00000e+00 -4.50208e+02 -3.63588e+02 -2.44128e+02
 -1.02690e+01  0.00000e+00  1.00000e-03  2.00000e-03  1.00000e-03]
[2.22655000e-04 6.10683852e-04 8.03737395e+06 3.05791234e+05
 9.05771599e+06 1.24401626e+06 5.03777283e-03 3.77467978e-03
 5.68719145e-03 6.37339419e-03]
[-2.22655000e-04 -6.10683852e-04 -8.03782416e+06 -3.06154822e+05
 -9.05796012e+06 -1.24402653e+06 -5.03777283e-03 -2.77467978e-03
 -3.68719145e-03 -5.37339419e-03]
